# Explore the pool audit (100K AceReason-1.1-SFT)

Interactive look at `analysis/pool_audit.parquet`: one row per pool sample, no text. Built by
`scripts/audit_pool.py` (Step 1); column meanings are in that script's docstring.

- Needs: `pandas`, `pyarrow`, `plotly` (and `ipywidgets` for the slice explorer at the end).
- The parquet is gitignored. Copy it from ARC:
  `scp <you>@tinkercliffs1.arc.vt.edu:~/acereasoning-sft100k/analysis/pool_audit.parquet analysis/`
- To read actual prompts/traces (section 8) you also need the pool text, `work/pool/pool.parquet`
  (from `scripts/fetch_data.py`). Everything else works without it.

Hover any chart for exact numbers; click legend entries to hide/show series.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

ROOT = Path.cwd() if (Path.cwd() / "analysis").exists() else Path.cwd().parent
AUDIT = ROOT / "analysis" / "pool_audit.parquet"
POOL_TEXT = ROOT / "work" / "pool" / "pool.parquet"
CUTOFF = 16384  # training cutoff_len

# Categorical colors, assigned in a fixed order and always following the entity (never re-cycled).
C = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
DOMAIN_COLOR = {"math": C[0], "code": C[1]}
MATH_SOURCES = ["OpenMathReasoning", "NuminaMath-CoT"]
CODE_SOURCES = ["OpenCodeReasoning", "opc-sft-stage2", "leetcode", "TACO", "MagicoderEvolInstruct", "apps"]
SOURCE_ORDER = MATH_SOURCES + CODE_SOURCES
SOURCE_COLOR = {s: C[i] for i, s in enumerate(SOURCE_ORDER)}
SEQ = "Blues"  # magnitude: one hue, light -> dark
INK, MUTED, GRID = "#0b0b0b", "#52514e", "#e6e5e0"

pio.templates["audit"] = go.layout.Template(layout=dict(
    font=dict(family="Inter, system-ui, sans-serif", size=13, color=INK),
    paper_bgcolor="white", plot_bgcolor="white",
    xaxis=dict(gridcolor=GRID, zeroline=False, linecolor=GRID, ticks=""),
    yaxis=dict(gridcolor=GRID, zeroline=False, linecolor=GRID, ticks=""),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, x=0),
    hoverlabel=dict(bgcolor="white", font_color=INK),
    margin=dict(l=60, r=30, t=70, b=50), bargap=0.15))
pio.templates.default = "audit"

audit = pd.read_parquet(AUDIT)
math = audit[audit.domain == "math"]
print(f"{len(audit):,} rows, {audit.shape[1]} columns  |  math {len(math):,}  code {(audit.domain == 'code').sum():,}")

## 1. What's in the pool

Rows per upstream source. Math is two sources; OpenMathReasoning alone is over half the pool.

In [ ]:
src = (audit.groupby(["domain", "source"]).size().rename("rows").reset_index()
       .assign(share=lambda d: d.rows / len(audit)))
fig = px.bar(src, x="rows", y="source", color="domain", orientation="h",
             category_orders={"source": SOURCE_ORDER}, color_discrete_map=DOMAIN_COLOR,
             custom_data=["share"], title="Rows per source")
fig.update_traces(marker_line_width=0, texttemplate="%{x:,}", textposition="outside", cliponaxis=False,
                  hovertemplate="%{y}<br>%{x:,} rows (%{customdata[0]:.1%})<extra></extra>")
fig.update_layout(height=380, yaxis_title=None, xaxis_title="rows", legend_title=None)
fig.show()

## 2. Length and the 16,384-token cutoff

Anything right of the dashed line gets its tail cut during training (usually including the `\boxed{}`
answer), which teaches the model to stop without answering.

In [ ]:
fig = px.histogram(audit, x="total_tokens", color="domain", nbins=120, barmode="overlay", opacity=0.75,
                   color_discrete_map=DOMAIN_COLOR, category_orders={"domain": ["math", "code"]},
                   title="Training-sequence length (prompt + response), by domain")
fig.add_vline(x=CUTOFF, line_dash="dash", line_color=MUTED,
              annotation_text="cutoff 16,384", annotation_position="top right", annotation_font_color=MUTED)
fig.update_layout(height=420, xaxis_title="total tokens (Qwen2.5 tokenizer)", yaxis_title="rows", legend_title=None)
fig.update_traces(hovertemplate="%{x} tokens<br>%{y:,} rows<extra></extra>")
fig.show()

In [ ]:
# Cumulative view per source: read "X% of this source fits under N tokens" directly.
fig = go.Figure()
for s in SOURCE_ORDER:
    v = np.sort(audit.loc[audit.source == s, "total_tokens"].to_numpy())
    if len(v) == 0:
        continue
    step = max(1, len(v) // 400)  # thin the curve; it is smooth at this resolution
    xs, ys = v[::step], (np.arange(len(v)) / len(v))[::step]
    fig.add_trace(go.Scatter(x=xs, y=ys, mode="lines", name=s, line=dict(width=2, color=SOURCE_COLOR[s]),
                             hovertemplate=s + "<br>%{y:.1%} of rows ≤ %{x:,} tokens<extra></extra>"))
fig.add_vline(x=CUTOFF, line_dash="dash", line_color=MUTED)
fig.update_layout(title="Cumulative share of rows by length, per source", height=460, hovermode="closest",
                  xaxis_title="total tokens", yaxis=dict(title="share of rows ≤ length", tickformat=".0%"))
fig.show()

In [ ]:
tr = (audit.groupby(["domain", "source"]).truncated.agg(["mean", "sum"]).reset_index()
      .sort_values("mean"))
fig = px.bar(tr, x="mean", y="source", color="domain", orientation="h", color_discrete_map=DOMAIN_COLOR,
             custom_data=["sum"], title="Share of rows truncated at 16,384, per source")
fig.update_traces(marker_line_width=0, texttemplate="%{x:.1%}", textposition="outside", cliponaxis=False,
                  hovertemplate="%{y}<br>%{x:.2%} truncated (%{customdata[0]:,} rows)<extra></extra>")
fig.update_layout(height=380, xaxis=dict(title=None, tickformat=".0%"), yaxis_title=None, legend_title=None)
fig.show()

## 3. The selection funnel (math)

How many math rows survive each basic filter, applied cumulatively in this order. This is the starting
point for Step 5; Step 4's pass-rate band will cut further.

In [ ]:
steps = [
    ("all math rows",            math.index == math.index),
    ("fits in 16,384 tokens",    ~math.truncated),
    ("has gradable \\boxed{}",   math.has_boxed),
    ("not in test set (contam)", ~math.contam),
    ("not in dev set",           ~math.contam_dev),
    ("no quality flag",          ~math.flagged),
    ("not proof-like",           ~math.proof_like),
]
keep = pd.Series(True, index=math.index)
rows = []
for name, mask in steps:
    keep &= mask
    rows.append({"step": name, "rows": int(keep.sum()), "unique prompts": math.loc[keep, "prompt_hash"].nunique()})
funnel = pd.DataFrame(rows)
funnel["dropped"] = (-funnel.rows.diff()).fillna(0).astype(int)

fig = go.Figure(go.Bar(x=funnel.rows, y=funnel.step, orientation="h", marker=dict(color=C[0], line_width=0),
                       text=[f"{r:,}" for r in funnel.rows], textposition="outside", cliponaxis=False,
                       customdata=funnel[["dropped", "unique prompts"]],
                       hovertemplate="%{y}<br>%{x:,} rows left (−%{customdata[0]:,})"
                                     "<br>%{customdata[1]:,} unique prompts<extra></extra>"))
fig.update_layout(title="Math rows surviving each filter (cumulative)", height=380,
                  yaxis=dict(autorange="reversed", title=None), xaxis_title="rows")
fig.show()
funnel

## 4. What kind of answers R1 gives (math)

`numeric` and `expression` answers can be graded in Step 4; `text` (prose in `\boxed{}`) and `none` cannot.

In [ ]:
kinds = ["numeric", "expression", "mc", "text", "none"]
ct = pd.crosstab(math.source, math.answer_kind).reindex(index=MATH_SOURCES, columns=kinds, fill_value=0)
share = ct.div(ct.sum(axis=1), axis=0)
fig = px.imshow(share, text_auto=".1%", color_continuous_scale=SEQ, aspect="auto",
                title="Answer kind by source (share of that source's rows)")
fig.update_traces(customdata=ct.to_numpy(),
                  hovertemplate="%{y} · %{x}<br>%{z:.1%} (%{customdata:,} rows)<extra></extra>")
fig.update_layout(height=300, coloraxis_showscale=False, xaxis_title=None, yaxis_title=None)
fig.show()
print("proof_like rows by source:"); print(math.groupby("source").proof_like.sum().to_string())

## 5. Duplicate prompts

Some problems appear several times, each with a different R1 trace. Step 4 uses these groups for
`r1_agreement` (do R1's answers to the same problem agree?). Note the y axis is log scale.

In [ ]:
dup = (audit.drop_duplicates("prompt_hash").groupby(["dup_count", "domain"]).size()
       .rename("prompts").reset_index())
fig = px.bar(dup, x="dup_count", y="prompts", color="domain", barmode="group", log_y=True,
             color_discrete_map=DOMAIN_COLOR, category_orders={"domain": ["math", "code"]},
             title="Responses per unique prompt")
fig.update_traces(marker_line_width=0, hovertemplate="%{y:,} prompts with %{x} responses<extra></extra>")
fig.update_layout(height=380, xaxis=dict(title="responses for the same prompt", dtick=1),
                  yaxis_title="unique prompts (log)", legend_title=None)
fig.show()

## 6. Contamination with the test benchmarks

Only rows with some overlap are shown (score > 0). Rows at ≥ 0.5 are excluded; the **0.3–0.5 band**
still needs a manual check (paraphrases score lower than verbatim copies).

In [ ]:
c = audit[audit.contam_score > 0]
fig = px.histogram(c, x="contam_score", nbins=50, log_y=True, color_discrete_sequence=[C[0]],
                   title=f"Overlap score with the closest test question ({len(c):,} rows with any overlap)")
fig.add_vrect(x0=0.3, x1=0.5, fillcolor=MUTED, opacity=0.12, line_width=0,
              annotation_text="check by hand", annotation_position="top left", annotation_font_color=MUTED)
fig.add_vline(x=0.5, line_dash="dash", line_color=MUTED,
              annotation_text="excluded ≥ 0.5", annotation_position="top right", annotation_font_color=MUTED)
fig.update_traces(marker_line_width=0, hovertemplate="score %{x}<br>%{y:,} rows<extra></extra>")
fig.update_layout(height=380, xaxis_title="share of the test question's 8-grams in the pool prompt",
                  yaxis_title="rows (log)")
fig.show()

near = audit[(audit.contam_score >= 0.3) & (audit.contam_score < 0.5)]
print(f"near-misses to check: {len(near)} rows")
display(near.groupby("contam_bench").size().rename("rows").to_frame().T)
near.sort_values("contam_score", ascending=False)[["pool_idx", "source", "contam_score", "contam_bench", "contam_qid"]].head(20)

## 7. Trace quality

Each point is one response (a 20K-row random sample, plus every flagged row). Low tail-compression ratio
= the end of the trace repeats itself. Check whether the few flagged rows really stand apart, or whether
unflagged rows sit just as low.

In [ ]:
samp = pd.concat([audit.sample(20000, random_state=0), audit[audit.flagged]]).drop_duplicates("pool_idx")
samp["status"] = np.where(samp.flagged, "flagged", "not flagged")
fig = px.scatter(samp, x="resp_tokens", y="tail_zlib_ratio", color="status", opacity=0.5,
                 color_discrete_map={"not flagged": C[0], "flagged": C[1]},
                 category_orders={"status": ["not flagged", "flagged"]},
                 hover_data={"pool_idx": True, "source": True, "max_line_repeat": True, "status": False},
                 title="Tail compressibility vs response length")
fig.update_traces(marker=dict(size=4, line_width=0), selector=dict(name="not flagged"))
fig.update_traces(marker=dict(size=10, line=dict(width=2, color="white")), selector=dict(name="flagged"))
fig.add_hline(y=0.10, line_dash="dash", line_color=MUTED,
              annotation_text="repetition threshold 0.10", annotation_font_color=MUTED)
fig.update_layout(height=460, xaxis_title="response tokens", yaxis_title="compressed / raw size (last 4 KB)",
                  legend_title=None)
fig.show()
print("lowest-ratio unflagged rows (worth reading):")
audit[~audit.flagged].nsmallest(10, "tail_zlib_ratio")[["pool_idx", "source", "resp_tokens", "tail_zlib_ratio", "max_line_repeat"]]

## 8. Slice explorer: filter, count, and read samples

Pick filters and see how many rows survive and what they look like. With `work/pool/pool.parquet`
present, `show(pool_idx)` prints the prompt, the start and end of R1's trace, and the extracted answer.

In [ ]:
_text = None

def pool_text():
    global _text
    if _text is None:
        if not POOL_TEXT.exists():
            raise FileNotFoundError(f"{POOL_TEXT} not found: run scripts/fetch_data.py, or copy it from ARC")
        _text = pd.read_parquet(POOL_TEXT, columns=["pool_idx", "instruction", "input", "output"]).set_index("pool_idx")
    return _text

def show(pool_idx, head=1500, tail=1500):
    a = audit.set_index("pool_idx").loc[pool_idx]
    t = pool_text().loc[pool_idx]
    prompt = t.instruction if not t.input else f"{t.instruction}\n{t.input}"
    print(f"pool_idx {pool_idx} | {a.source} | {a.total_tokens:,} tokens | truncated={a.truncated} "
          f"| answer_kind={a.answer_kind} | dup_count={a.dup_count} | contam={a.contam_score}")
    print("=" * 100, "\nPROMPT\n", prompt)
    out = t.output
    print("=" * 100, f"\nRESPONSE ({len(out):,} chars)\n", out[:head])
    if len(out) > head + tail:
        print(f"\n[... {len(out) - head - tail:,} chars omitted ...]\n")
        print(out[-tail:])
    print("=" * 100, "\nR1 ANSWER:", a.r1_answer)

In [ ]:
import ipywidgets as W
from IPython.display import clear_output, display

w_domain = W.Dropdown(options=["math", "code", "all"], value="math", description="domain")
w_source = W.SelectMultiple(options=SOURCE_ORDER, value=tuple(MATH_SOURCES), description="source", rows=5)
w_kind = W.SelectMultiple(options=["numeric", "expression", "mc", "text", "none"],
                          value=("numeric", "expression", "mc"), description="answer", rows=5)
w_len = W.IntRangeSlider(value=(0, CUTOFF), min=0, max=int(audit.total_tokens.max()), step=256,
                         description="tokens", continuous_update=False, layout=W.Layout(width="520px"))
w_clean = W.Checkbox(value=True, description="drop contam / dev / flagged")
w_proof = W.Checkbox(value=True, description="drop proof-like")
w_dup = W.IntRangeSlider(value=(1, 11), min=1, max=11, description="dup_count", continuous_update=False)
out = W.Output()

def current_slice():
    d = audit
    if w_domain.value != "all":
        d = d[d.domain == w_domain.value]
    d = d[d.source.isin(w_source.value) & d.total_tokens.between(*w_len.value) & d.dup_count.between(*w_dup.value)]
    if w_domain.value == "math":
        d = d[d.answer_kind.isin(w_kind.value)]
    if w_clean.value:
        d = d[~d.contam & ~d.contam_dev & ~d.flagged]
    if w_proof.value:
        d = d[~d.proof_like]
    return d

def refresh(*_):
    with out:
        clear_output(wait=True)
        d = current_slice()
        print(f"{len(d):,} rows · {d.prompt_hash.nunique():,} unique prompts · "
              f"{len(d) / 15000:.2f}× a 15K subset · median {int(d.total_tokens.median()) if len(d) else 0:,} tokens")
        if len(d):
            fig = px.histogram(d, x="total_tokens", color="source", nbins=80, height=300,
                               color_discrete_map=SOURCE_COLOR, category_orders={"source": SOURCE_ORDER})
            fig.update_traces(marker_line_width=0)
            fig.update_layout(margin=dict(t=30), xaxis_title="total tokens", yaxis_title="rows", legend_title=None)
            fig.show()
            display(d.sample(min(10, len(d)), random_state=0)[
                ["pool_idx", "source", "total_tokens", "answer_kind", "r1_answer", "dup_count"]])

for w in [w_domain, w_source, w_kind, w_len, w_clean, w_proof, w_dup]:
    w.observe(refresh, "value")
display(W.VBox([W.HBox([w_domain, w_clean, w_proof]), W.HBox([w_source, w_kind]), w_len, w_dup, out]))
refresh()

In [ ]:
# Read a sample from the slice above (needs work/pool/pool.parquet). Replace with any pool_idx.
s = current_slice()
if len(s) and POOL_TEXT.exists():
    show(int(s.sample(1).pool_idx.iloc[0]))
else:
    print("no rows in the slice, or pool text not available locally")